# NLP pe rețete: de la text la features pentru Machine Learning

În notebook-ul anterior am încărcat rețetele cu `requests` și am lucrat cu coloane numerice.

Aici ne concentrăm pe coloanele **cu cuvinte**:
1. `name`
2. `ingredients`
3. `instructions`
4. `tags`
5. `mealType`

Scopul: transformăm textul în **numere** pe care un model Scikit-learn le poate folosi.

## 1. Încărcarea datelor (ca în notebook-ul anterior)

In [1]:
import requests
import pandas as pd
import numpy as np

URL = "https://dummyjson.com/recipes"
response = requests.get(URL)
json_response = response.json()

recipes_df = pd.DataFrame(json_response["recipes"])
recipes_df.head(3)

,id,name,ingredients,instructions,prepTimeMinutes,cookTimeMinutes,servings,difficulty,cuisine,caloriesPerServing,tags,userId,image,rating,reviewCount,mealType
0,1,Classic Margherita Pizza,"[Pizza dough, Tomato sauce, Fresh mozzarella c...","[Preheat the oven to 475°F (245°C)., Roll out ...",20,15,4,Easy,Italian,300,"[Pizza, Italian]",166,https://cdn.dummyjson.com/recipe-images/1.webp,4.6,98,[Dinner]
1,2,Vegetarian Stir-Fry,"[Tofu, cubed, Broccoli florets, Carrots, slice...","[In a wok, heat sesame oil over medium-high he...",15,20,3,Medium,Asian,250,"[Vegetarian, Stir-fry, Asian]",143,https://cdn.dummyjson.com/recipe-images/2.webp,4.7,26,[Lunch]
2,3,Chocolate Chip Cookies,"[All-purpose flour, Butter, softened, Brown su...","[Preheat the oven to 350°F (175°C)., In a bowl...",15,10,24,Easy,American,150,"[Cookies, Dessert, Baking]",34,https://cdn.dummyjson.com/recipe-images/3.webp,4.9,13,"[Snack, Dessert]"


In [2]:
text_cols = ["name", "ingredients", "instructions", "tags", "mealType"]
recipes_df[text_cols].head(3)

,name,ingredients,instructions,tags,mealType
0,Classic Margherita Pizza,"[Pizza dough, Tomato sauce, Fresh mozzarella c...","[Preheat the oven to 475°F (245°C)., Roll out ...","[Pizza, Italian]",[Dinner]
1,Vegetarian Stir-Fry,"[Tofu, cubed, Broccoli florets, Carrots, slice...","[In a wok, heat sesame oil over medium-high he...","[Vegetarian, Stir-fry, Asian]",[Lunch]
2,Chocolate Chip Cookies,"[All-purpose flour, Butter, softened, Brown su...","[Preheat the oven to 350°F (175°C)., In a bowl...","[Cookies, Dessert, Baking]","[Snack, Dessert]"


In [3]:
# Tipurile de date: liste în celule, nu string-uri simple
for col in text_cols:
    print(f"{col:15} tip primul element: {type(recipes_df[col].iloc[0])}")

name            tip primul element: <class 'str'>
ingredients     tip primul element: <class 'list'>
instructions    tip primul element: <class 'list'>
tags            tip primul element: <class 'list'>
mealType        tip primul element: <class 'list'>


## 2. Pregătirea textului: liste → string

`ingredients`, `instructions`, `tags` și `mealType` sunt **liste**.
Pentru NLP (și pentru majoritatea encoder-elor) le transformăm în text simplu.

In [4]:
def list_to_text(value):
    """Unește elementele unei liste într-un singur string."""
    if isinstance(value, list):
        return " ".join(str(item) for item in value)
    return str(value)

nlp_df = recipes_df.copy()

nlp_df["ingredients_text"] = nlp_df["ingredients"].apply(list_to_text)
nlp_df["instructions_text"] = nlp_df["instructions"].apply(list_to_text)
nlp_df["tags_text"] = nlp_df["tags"].apply(list_to_text)
nlp_df["mealType_text"] = nlp_df["mealType"].apply(list_to_text)

nlp_df[["name", "ingredients_text", "tags_text", "mealType_text"]].head(3)

,name,ingredients_text,tags_text,mealType_text
0,Classic Margherita Pizza,Pizza dough Tomato sauce Fresh mozzarella chee...,Pizza Italian,Dinner
1,Vegetarian Stir-Fry,"Tofu, cubed Broccoli florets Carrots, sliced B...",Vegetarian Stir-fry Asian,Lunch
2,Chocolate Chip Cookies,"All-purpose flour Butter, softened Brown sugar...",Cookies Dessert Baking,Snack Dessert


---
## 3. Encodări pentru `tags` și `mealType`

Aceste coloane sunt **categorii** (etichete), uneori multiple pe același rând.

Vrem să le transformăm în coloane numerice — stil **One-Hot**.

### 3.1 One-Hot cu `pandas.get_dummies` (mealType)

Dacă fiecare rețetă ar avea un singur tip de masă, `get_dummies` e suficient.
La noi `mealType` e listă — mai întâi o „explodăm” sau luăm primul tip.

In [5]:
# Varianta simplă: primul tip de masă ca o singură categorie
nlp_df["mealType_primary"] = nlp_df["mealType"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Unknown"
)

meal_dummies = pd.get_dummies(nlp_df["mealType_primary"], prefix="meal")
meal_dummies.head()

,meal_Appetizer,meal_Beverage,meal_Breakfast,meal_Dessert,meal_Dinner,meal_Lunch,meal_Snack
0,False,False,False,False,True,False,False
1,False,False,False,False,False,True,False
2,False,False,False,False,False,False,True
3,False,False,False,False,False,True,False
4,False,False,False,False,True,False,False


In [6]:
# Lipim One-Hot-ul înapoi în tabel
df_with_meal = pd.concat([nlp_df[["name", "mealType_primary"]], meal_dummies], axis=1)
df_with_meal.head()

,name,mealType_primary,meal_Appetizer,meal_Beverage,meal_Breakfast,meal_Dessert,meal_Dinner,meal_Lunch,meal_Snack
0,Classic Margherita Pizza,Dinner,False,False,False,False,True,False,False
1,Vegetarian Stir-Fry,Lunch,False,False,False,False,False,True,False
2,Chocolate Chip Cookies,Snack,False,False,False,False,False,False,True
3,Chicken Alfredo Pasta,Lunch,False,False,False,False,False,True,False
4,Mango Salsa Chicken,Dinner,False,False,False,False,True,False,False


### 3.2 One-Hot / Multi-Hot cu `MultiLabelBinarizer` (tags & mealType)

Când o rețetă are **mai multe** etichete (`['Pizza', 'Italian']`), One-Hot clasic pe un singur label nu ajunge.

`MultiLabelBinarizer` creează o coloană pentru fiecare valoare posibilă și pune `1` dacă e prezentă.

In [7]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_tags = MultiLabelBinarizer()
tags_encoded = mlb_tags.fit_transform(nlp_df["tags"])

tags_ohe = pd.DataFrame(
    tags_encoded,
    columns=[f"tag_{c}" for c in mlb_tags.classes_],
    index=nlp_df.index,
)

print("Număr de tag-uri unice:", len(mlb_tags.classes_))
print("Exemple:", list(mlb_tags.classes_[:10]))
tags_ohe.head()

Număr de tag-uri unice: 61
Exemple: ['Asian', 'Baking', 'Banana', 'Beef', 'Bibimbap', 'Biryani', 'Blueberry', 'Borscht', 'Brazilian', 'Bruschetta']


,tag_Asian,tag_Baking,tag_Banana,tag_Beef,tag_Bibimbap,tag_Biryani,tag_Blueberry,tag_Borscht,tag_Brazilian,tag_Bruschetta,...,tag_Smoothie,tag_Soup,tag_Stir-fry,tag_Street food,tag_Tagine,tag_Thai,tag_Tiramisu,tag_Turkish,tag_Vegetarian,tag_Wrap
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
mlb_meal = MultiLabelBinarizer()
meal_encoded = mlb_meal.fit_transform(nlp_df["mealType"])

meal_ohe = pd.DataFrame(
    meal_encoded,
    columns=[f"meal_{c}" for c in mlb_meal.classes_],
    index=nlp_df.index,
)

print("Tipuri de masă:", list(mlb_meal.classes_))
meal_ohe.head()

Tipuri de masă: ['Appetizer', 'Beverage', 'Breakfast', 'Dessert', 'Dinner', 'Lunch', 'Side Dish', 'Snack', 'Snacks']


,meal_Appetizer,meal_Beverage,meal_Breakfast,meal_Dessert,meal_Dinner,meal_Lunch,meal_Side Dish,meal_Snack,meal_Snacks
0,0,0,0,0,1,0,0,0,0
1,0,0,0,0,0,1,0,0,0
2,0,0,0,1,0,0,0,1,0
3,0,0,0,0,1,1,0,0,0
4,0,0,0,0,1,0,0,0,0


### 3.3 Alte tipuri de encodare (pandas + sklearn)

| Metodă | Ce face | Când e utilă |
|--------|---------|--------------|
| **One-Hot / Multi-Hot** | o coloană 0/1 per categorie | categorii fără ordine |
| **Label Encoding** | un număr întreg per categorie | rareori pentru ML clasic (impune o ordine falsă) |
| **Ordinal Encoding** | numere cu ordine reală | Easy < Medium < Hard |
| **Count / Frequency** | câte apariții / frecvența | multe categorii rare |
| **Target Encoding** | media țintei pe categorie | categorii cu multe valori (atenție la leakage) |

#### Label Encoding (sklearn)

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
nlp_df["mealType_label"] = le.fit_transform(nlp_df["mealType_primary"])

pd.DataFrame({
    "mealType_primary": nlp_df["mealType_primary"],
    "mealType_label": nlp_df["mealType_label"],
}).drop_duplicates().sort_values("mealType_label")

,mealType_primary,mealType_label
6,Appetizer,0
21,Beverage,1
14,Breakfast,2
22,Dessert,3
0,Dinner,4
1,Lunch,5
2,Snack,6


#### Ordinal Encoding (ordine reală — ex. difficulty)

In [10]:
from sklearn.preprocessing import OrdinalEncoder

# Ordinea contează: Easy < Medium < Hard
ord_enc = OrdinalEncoder(categories=[["Easy", "Medium", "Hard"]])
nlp_df["difficulty_ord"] = ord_enc.fit_transform(nlp_df[["difficulty"]])

nlp_df[["difficulty", "difficulty_ord"]].drop_duplicates()

,difficulty,difficulty_ord
0,Easy,0.0
1,Medium,1.0


#### Frequency Encoding (pandas)

In [11]:
freq = nlp_df["mealType_primary"].value_counts(normalize=True)
nlp_df["mealType_freq"] = nlp_df["mealType_primary"].map(freq)

nlp_df[["mealType_primary", "mealType_freq"]].drop_duplicates().sort_values(
    "mealType_freq", ascending=False
)

,mealType_primary,mealType_freq
0,Dinner,0.400000
1,Lunch,0.300000
14,Breakfast,0.100000
2,Snack,0.066667
21,Beverage,0.066667
6,Appetizer,0.033333
22,Dessert,0.033333


#### Mapare manuală (ca în notebook-ul 01)

In [12]:
nlp_df["difficulty_mapped"] = nlp_df["difficulty"].map({"Easy": 0, "Medium": 50, "Hard": 100})
nlp_df[["difficulty", "difficulty_mapped", "difficulty_ord"]].head()

,difficulty,difficulty_mapped,difficulty_ord
0,Easy,0,0.0
1,Medium,50,1.0
2,Easy,0,0.0
3,Medium,50,1.0
4,Easy,0,0.0


---
## 4. Text liber: `name`, `ingredients`, `instructions`

Aici nu avem categorii fixe, ci **propoziții / liste de cuvinte**.
Transformăm textul în vectori numerici cu tehnici tipice NLP:

### 4.1 Feature-uri simple derivate din text (lungime, număr de cuvinte)

In [13]:
nlp_df["name_len"] = nlp_df["name"].str.len()
nlp_df["name_words"] = nlp_df["name"].str.split().str.len()

nlp_df["n_ingredients"] = nlp_df["ingredients"].apply(len)
nlp_df["n_instructions"] = nlp_df["instructions"].apply(len)
nlp_df["ingredients_chars"] = nlp_df["ingredients_text"].str.len()
nlp_df["instructions_chars"] = nlp_df["instructions_text"].str.len()

nlp_df[[
    "name", "name_words", "n_ingredients", "n_instructions",
    "ingredients_chars", "instructions_chars"
]].head()

,name,name_words,n_ingredients,n_instructions,ingredients_chars,instructions_chars
0,Classic Margherita Pizza,3,6,6,102,309
1,Vegetarian Stir-Fry,2,9,6,140,307
2,Chocolate Chip Cookies,3,9,8,112,482
3,Chicken Alfredo Pasta,3,8,7,148,367
4,Mango Salsa Chicken,3,8,5,148,260


### 4.2 Bag of Words — `CountVectorizer`

Numără de câte ori apare fiecare cuvânt. Rezultatul e o matrice **rară** (multe zerouri).

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer(stop_words="english", max_features=20)
name_bow = count_vec.fit_transform(nlp_df["name"])

name_bow_df = pd.DataFrame(
    name_bow.toarray(),
    columns=[f"bow_{w}" for w in count_vec.get_feature_names_out()],
    index=nlp_df.index,
)

print("Vocabular (top 20 cuvinte din name):")
print(list(count_vec.get_feature_names_out()))
pd.concat([nlp_df[["name"]], name_bow_df], axis=1).head(3)

Vocabular (top 20 cuvinte din name):
['chicken', 'fry', 'kebabs', 'lebanese', 'makhani', 'makki', 'mango', 'margherita', 'masala', 'mexican', 'moroccan', 'moussaka', 'murgh', 'pasta', 'pizza', 'quinoa', 'ramen', 'russian', 'salad', 'stir']


,name,bow_chicken,bow_fry,bow_kebabs,bow_lebanese,bow_makhani,bow_makki,bow_mango,bow_margherita,bow_masala,...,bow_moroccan,bow_moussaka,bow_murgh,bow_pasta,bow_pizza,bow_quinoa,bow_ramen,bow_russian,bow_salad,bow_stir
0,Classic Margherita Pizza,0,0,0,0,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
1,Vegetarian Stir-Fry,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,Chocolate Chip Cookies,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 4.3 TF-IDF — `TfidfVectorizer`

**TF-IDF** (Term Frequency – Inverse Document Frequency) dă greutate mai mare cuvintelor
specifice unei rețete și mai mică celor foarte comune (ex. „and”, „the” — deja eliminate cu `stop_words`).

E adesea mai bun decât Bag of Words pentru clasificare.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_name = TfidfVectorizer(stop_words="english", max_features=30)
name_tfidf = tfidf_name.fit_transform(nlp_df["name"])

name_tfidf_df = pd.DataFrame(
    name_tfidf.toarray(),
    columns=[f"tfidf_name_{w}" for w in tfidf_name.get_feature_names_out()],
    index=nlp_df.index,
)
name_tfidf_df.head(3)

,tfidf_name_chicken,tfidf_name_fry,tfidf_name_kebabs,tfidf_name_lassi,tfidf_name_lebanese,tfidf_name_makhani,tfidf_name_makki,tfidf_name_mango,tfidf_name_margherita,tfidf_name_masala,...,tfidf_name_salad,tfidf_name_soup,tfidf_name_spinach,tfidf_name_stir,tfidf_name_street,tfidf_name_tagine,tfidf_name_thai,tfidf_name_tiramisu,tfidf_name_tomato,tfidf_name_turkish
0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.707107,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.707107,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# Același lucru pe ingredients și instructions
tfidf_ing = TfidfVectorizer(stop_words="english", max_features=40)
ing_tfidf = tfidf_ing.fit_transform(nlp_df["ingredients_text"])

tfidf_instr = TfidfVectorizer(stop_words="english", max_features=40)
instr_tfidf = tfidf_instr.fit_transform(nlp_df["instructions_text"])

print("Vocabular ingredients (primele 15):", list(tfidf_ing.get_feature_names_out()[:15]))
print("Vocabular instructions (primele 15):", list(tfidf_instr.get_feature_names_out()[:15]))
print("Shape ingredients TF-IDF:", ing_tfidf.shape)
print("Shape instructions TF-IDF:", instr_tfidf.shape)

Vocabular ingredients (primele 15): ['beef', 'cheese', 'chicken', 'chili', 'chilies', 'chopped', 'cooked', 'coriander', 'cream', 'cumin', 'diced', 'finely', 'fresh', 'garlic', 'ginger']
Vocabular instructions (primele 15): ['add', 'bowl', 'brown', 'chicken', 'chopped', 'combine', 'cook', 'cooked', 'diced', 'drizzle', 'enjoy', 'fresh', 'garlic', 'ginger', 'golden']
Shape ingredients TF-IDF: (30, 40)
Shape instructions TF-IDF: (30, 40)


### 4.4 N-grame (bi-grame)

Uneori perechile de cuvinte (`"olive oil"`, `"tomato sauce"`) sunt mai informative decât cuvintele izolate.

In [17]:
tfidf_bigrams = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),  # unigrame + bigrame
    max_features=25,
)
ing_bigrams = tfidf_bigrams.fit_transform(nlp_df["ingredients_text"])

print("Feature-uri (uni + bi-grame):")
print(list(tfidf_bigrams.get_feature_names_out()))

Feature-uri (uni + bi-grame):
['beef', 'chicken', 'chopped', 'coriander', 'diced', 'finely', 'finely chopped', 'fresh', 'garlic', 'garlic minced', 'ginger', 'leaves', 'minced', 'oil', 'onions', 'pepper', 'pepper taste', 'powder', 'red', 'salt', 'salt pepper', 'sauce', 'sliced', 'taste', 'tomatoes']


---
## 5. Construim matricea de features pentru ML

Combinăm:
- feature-uri numerice originale
- One-Hot / Multi-Hot pe `tags` și `mealType`
- feature-uri simple din text
- TF-IDF pe `name` + `ingredients` (+ opțional instructions)

**Ținta (y):** `difficulty` — clasificare Easy vs Medium.

In [18]:
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

# --- ținta ---
y = nlp_df["difficulty"]
print("Distribuție difficulty:")
print(y.value_counts())

Distribuție difficulty:
difficulty
Medium    16
Easy      14
Name: count, dtype: int64


In [19]:
# Feature-uri numerice (scalate)
numeric_cols = [
    "prepTimeMinutes", "cookTimeMinutes", "servings",
    "caloriesPerServing", "rating", "reviewCount",
    "name_words", "n_ingredients", "n_instructions",
    "ingredients_chars", "instructions_chars",
]

scaler = StandardScaler()
X_numeric = scaler.fit_transform(nlp_df[numeric_cols])

# TF-IDF pe name + ingredients (instructions e lung; îl lăsăm opțional)
tfidf_combo = TfidfVectorizer(stop_words="english", max_features=50)
X_text = tfidf_combo.fit_transform(
    nlp_df["name"] + " " + nlp_df["ingredients_text"]
)

# Multi-Hot tags + mealType
X_tags = mlb_tags.transform(nlp_df["tags"])
X_meal = mlb_meal.transform(nlp_df["mealType"])

# Matricea finală (mixtă: densă + sparse)
X = hstack([
    csr_matrix(X_numeric),
    X_text,
    csr_matrix(X_tags),
    csr_matrix(X_meal),
])

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (30, 131)
Shape y: (30,)


---
## 6. Model clasic Scikit-learn

Cu doar 30 de rețete, rezultatele sunt **illustrative** (set mic).
Folosim `LogisticRegression` — un model clasic, rapid, potrivit pentru features TF-IDF + One-Hot.

In [20]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy pe test:", clf.score(X_test, y_test))
print("\nClassification report:")
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy pe test: 0.7777777777777778

Classification report:
              precision    recall  f1-score   support

        Easy       1.00      0.50      0.67         4
      Medium       0.71      1.00      0.83         5

    accuracy                           0.78         9
   macro avg       0.86      0.75      0.75         9
weighted avg       0.84      0.78      0.76         9

Confusion matrix:
[[2 2]
 [0 5]]


In [21]:
# Cross-validation pe tot setul (mai stabil pe date mici)
cv_scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
print("CV accuracy (5-fold):", cv_scores)
print("Media:", round(cv_scores.mean(), 3), "±", round(cv_scores.std(), 3))

CV accuracy (5-fold): [0.5        1.         0.66666667 1.         0.83333333]
Media: 0.8 ± 0.194


### Alternativă: Random Forest

Pe features mixte (numerice + sparse text), Random Forest e o alternativă clasică.
Convertim la densă doar pentru demonstrație (OK pe set mic).

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train.toarray(), y_train)

print("RF accuracy pe test:", rf.score(X_test.toarray(), y_test))
print(classification_report(y_test, rf.predict(X_test.toarray())))

RF accuracy pe test: 0.7777777777777778
              precision    recall  f1-score   support

        Easy       1.00      0.50      0.67         4
      Medium       0.71      1.00      0.83         5

    accuracy                           0.78         9
   macro avg       0.86      0.75      0.75         9
weighted avg       0.84      0.78      0.76         9



---
## 7. Pipeline end-to-end (recomandat în practică)

`Pipeline` + `ColumnTransformer` țin împreună preprocessing-ul și modelul — evită data leakage.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# DataFrame „curat” pentru pipeline
pipe_df = nlp_df[[
    "name", "ingredients_text", "prepTimeMinutes", "cookTimeMinutes",
    "servings", "caloriesPerServing", "rating", "reviewCount",
    "n_ingredients", "difficulty",
]].copy()

# Text combinat name + ingredients
pipe_df["text"] = pipe_df["name"] + " " + pipe_df["ingredients_text"]

X_pipe = pipe_df.drop(columns=["difficulty", "name", "ingredients_text"])
y_pipe = pipe_df["difficulty"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), [
        "prepTimeMinutes", "cookTimeMinutes", "servings",
        "caloriesPerServing", "rating", "reviewCount", "n_ingredients",
    ]),
    ("text", TfidfVectorizer(stop_words="english", max_features=40), "text"),
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

Xtr, Xte, ytr, yte = train_test_split(
    X_pipe, y_pipe, test_size=0.3, random_state=42, stratify=y_pipe
)

model.fit(Xtr, ytr)
print("Pipeline accuracy:", model.score(Xte, yte))
print(classification_report(yte, model.predict(Xte)))

Pipeline accuracy: 0.7777777777777778
              precision    recall  f1-score   support

        Easy       1.00      0.50      0.67         4
      Medium       0.71      1.00      0.83         5

    accuracy                           0.78         9
   macro avg       0.86      0.75      0.75         9
weighted avg       0.84      0.78      0.76         9



---
## Rezumat

| Coloană | Tip | Tehnici potrivite |
|---------|-----|-------------------|
| `tags`, `mealType` | categorii (multi-label) | `MultiLabelBinarizer`, `get_dummies`, Label / Frequency encoding |
| `difficulty` | categorie ordonată | `OrdinalEncoder`, `.map()` |
| `name`, `ingredients`, `instructions` | text liber | lungimi, `CountVectorizer`, `TfidfVectorizer`, n-grame |
| totul împreună | features mixte | `hstack` / `ColumnTransformer` + LogisticRegression / RandomForest |

Pași tipici NLP → ML:
1. Curățare / unire liste → string
2. Encoding categorii (One-Hot / Multi-Hot)
3. Vectorizare text (BoW / TF-IDF)
4. Scalare numerice
5. Antrenare model Scikit-learn